In [1]:
import sqlite3
from functions.pred import *
from functions.xai import *
from functions.eval import *
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, jaccard_score

In [7]:
conn = sqlite3.connect('data/xs2a_db.sqlite3')
c = conn.cursor()

In [8]:
polarite_map = {1: 'positive', 2: 'neutral', 3: 'negative'}

In [9]:
comments = {}
c.execute('SELECT * FROM myApp_datasetcommentaire')
rows = c.fetchall()
for row in rows:
    comment_content = c.execute('SELECT * FROM myApp_commentaire WHERE id=' + str(row[0])).fetchall()[0][1]
    comments[row[0]] = {"commentaire": comment_content, "polarite": polarite_map[row[2]]}

In [10]:
annotations = []
c.execute('SELECT * FROM myApp_annotation')
rows = c.fetchall()
for row in rows:
    annotations.append({"id": row[3], "mot": row[1]})

In [11]:
# number of annotations words
len(annotations)

1980

In [12]:
annotated_comments = set()
for annotation in annotations:
    annotated_comments.add(annotation["id"])

In [13]:
# number of annotated comments
len(annotated_comments)

303

In [14]:
# model_pred_col = "Camelbert-MSA"
# model_name = "CAMeL-Lab/bert-base-arabic-camelbert-msa-sentiment"
model_name = "PRAli22/AraBert-Arabic-Sentiment-Analysis"
model_pred_col = "AraBert"

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_index = 0 if torch.cuda.is_available() else -1
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_attentions=True).to(device)
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device_index, top_k=None)
polarities = list(model.config.id2label.values())

Device set to use cuda:0


In [16]:
lime_num_samples = 100
shap_max_evals = 100
ig_n_steps = 50

In [17]:
lime_explainer = LimeExplainer(model_name, device, num_samples=lime_num_samples)
shap_explainer = ShapExplainer(model_name, device, max_evals=shap_max_evals)
ig_explainer = IgExplainer(model_name, device, n_steps=ig_n_steps)
dl_explainer = DeepLiftExplainer(model_name, device)
ensemble_explainer = EnsembleExplainer(mean=True, median=True)

In [18]:
def token_in_annotated_mots(token, annotated_mots):
    if token.startswith("##"):
        token = token[2:]
    for annotated_mot in annotated_mots:
        if token in annotated_mot:
            return True
    return False

In [19]:
for comment_id in annotated_comments:
    comment_text = comments[comment_id]["commentaire"]
    comment_text_preprocessed = remove_chaklas(comment_text)
    comments[comment_id]["commentaire"] = comment_text_preprocessed

In [20]:
df = pd.DataFrame(columns=["comment_id", "token", "polarity", "lime_weight", "shap_weight", "ig_weight", "deeplift_weight", "human_annot"])

In [27]:
if model_name == "PRAli22/AraBert-Arabic-Sentiment-Analysis":
    # the first letter of each polarity in the AraBert model is uppercase
    # we need to convert the polarities of the human annotations to uppercase in comments dictionary
    for comment_id in annotated_comments:
        comments[comment_id]["polarite"] = comments[comment_id]["polarite"].capitalize()

In [29]:
for comment_id in tqdm(annotated_comments, desc="Explaining comments", unit="comment"):
    lime_res = lime_explainer.explain(comments[comment_id]["commentaire"],
        comments[comment_id]["polarite"])
    
    shap_res = shap_explainer.explain(comments[comment_id]["commentaire"],
        comments[comment_id]["polarite"])

    ig_res = ig_explainer.explain(comments[comment_id]["commentaire"],
        comments[comment_id]["polarite"])

    deeplift_res = dl_explainer.explain(comments[comment_id]["commentaire"],
        comments[comment_id]["polarite"])

    exai4_mean, exai4_median = ensemble_explainer.explain(lime_results=lime_res, shap_results=shap_res,
        ig_results=ig_res, dl_results=deeplift_res)

    exai3_mean, exai3_median = ensemble_explainer.explain(lime_results=lime_res, shap_results=shap_res,
        ig_results=ig_res, dl_results=None)

    exai2_mean, exai2_median = ensemble_explainer.explain(lime_results=lime_res, shap_results=shap_res,
        ig_results=None, dl_results=None)

    annotated_mots = {annotation["mot"]
        for annotation in annotations if annotation["id"] == comment_id}

    tokens = [r[1] for r in lime_res]

    for i, token in enumerate(tokens):
        human_annot = int(token_in_annotated_mots(token, annotated_mots))

        df = pd.concat([df,
            pd.DataFrame({
                "comment_id": [comment_id],
                "token": [token],
                "polarity": [comments[comment_id]["polarite"]],
                "lime_weight": [lime_res[i][2]],
                "shap_weight": [shap_res[i][2]],
                "ig_weight": [ig_res[i][2]],
                "deeplift_weight": [deeplift_res[i][2]],
                "exai4_mean": [exai4_mean[i][2]],
                "exai4_median": [exai4_median[i][2]],
                "exai3_mean": [exai3_mean[i][2]],
                "exai3_median": [exai3_median[i][2]],
                "exai2_mean": [exai2_mean[i][2]],
                "exai2_median": [exai2_median[i][2]],
                "human_annot": [human_annot]
            })], ignore_index=True)

Explaining comments:   0%|          | 0/303 [00:00<?, ?comment/s]

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\captum\_utils\gradient.py:57: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\captum\attr\_core\deep_lift.py:304: UserWarning: Setting forward, backward hooks and attributes on non-linear
               activations. The hooks and attributes will be removed
            after the attribution is finished
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\captum\_utils\gradient.py:57: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\captum\attr\_core\deep_lift.py:304: UserWarning: Setting forward, backward hooks and attributes on non-linear
               activations. The hooks and attributes w

In [30]:
df

,comment_id,token,polarity,lime_weight,shap_weight,ig_weight,deeplift_weight,human_annot,exai4_mean,exai4_median,exai3_mean,exai3_median,exai2_mean,exai2_median
0,1024,فندق,Positive,0.002522,0.00305,1.0,0.388327,0,0.348475,0.195688,0.335191,0.003050,0.002786,0.002786
1,1024,يناسب,Positive,0.433623,-0.303223,0.247639,1.0,0,0.344510,0.340631,0.126013,0.247639,0.065200,0.065200
2,1024,إمكانيات,Positive,-0.483691,-0.303223,-0.066209,-0.530133,1,-0.345814,-0.393457,-0.284374,-0.303223,-0.393457,-0.393457
3,1024,##ك,Positive,-0.530191,-0.509223,-0.125183,-0.506929,1,-0.417882,-0.508076,-0.388199,-0.509223,-0.519707,-0.519707
4,1024,إن,Positive,-0.452711,-0.509223,0.097086,-0.464861,0,-0.332427,-0.458786,-0.288282,-0.452711,-0.480967,-0.480967
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15917,1023,مرتفع,Negative,-0.695771,-0.115274,-0.762951,0.272026,0,-0.325493,-0.405523,-0.524666,-0.695771,-0.405523,-0.405523
15918,1023,##ه,Negative,-0.311345,-0.283067,-0.775175,0.062875,1,-0.326678,-0.297206,-0.456529,-0.311345,-0.297206,-0.297206
15919,1023,جدا,Negative,-0.349075,-0.670853,-0.337745,-0.049466,0,-0.351785,-0.343410,-0.452558,-0.349075,-0.509964,-0.509964
15920,1023,ومبالغ,Negative,-0.34814,1.0,-0.211196,-0.797707,1,-0.089261,-0.279668,0.146888,-0.211196,0.325930,0.325930


In [31]:
def top_tokens_elbow(token_scores):
    # Sort by importance descending
    sorted_token_weights = sorted(token_scores, key=lambda x: x[1], reverse=True)
    tokens = [x[0] for x in sorted_token_weights]
    scores = np.array([x[1] for x in sorted_token_weights])

    n = scores.size
    if n < 2:
        elbow_idx = 1
    else:
        x = np.arange(n, dtype=np.float64)
        x_mean = (n - 1) / 2.0
        y_mean = scores.mean()

        dx = x - x_mean
        slope = np.dot(dx, scores - y_mean) / np.dot(dx, dx)
        intercept = y_mean - slope * x_mean

        fitted = slope * x + intercept
        residuals = fitted - scores
        elbow_idx = np.argmax(residuals[1:]) + 1

    if elbow_idx is None:
        top_tokens = []  # No elbow detected
    else:
        threshold = scores[elbow_idx - 1]  # Adjust for 0-based index
        top_tokens = [tok for tok, score in zip(tokens, scores) if score >= threshold]
    return top_tokens

In [32]:
df.columns

Index(['comment_id', 'token', 'polarity', 'lime_weight', 'shap_weight',
       'ig_weight', 'deeplift_weight', 'human_annot', 'exai4_mean',
       'exai4_median', 'exai3_mean', 'exai3_median', 'exai2_mean',
       'exai2_median'],
      dtype='str')

In [33]:
df["lime_hard"] = 0
df["shap_hard"] = 0
df["ig_hard"] = 0
df["deeplift_hard"] = 0
df["exai4_mean_hard"] = 0
df["exai4_median_hard"] = 0
df["exai3_mean_hard"] = 0
df["exai3_median_hard"] = 0
df["exai2_mean_hard"] = 0

for comment_id in tqdm(annotated_comments, desc="Running elbow on explanations", unit="comment"):
    # Extract token-weight pairs for each method
    lime_token_scores = df[df["comment_id"] == comment_id][["token", "lime_weight"]].to_dict(orient="records")
    shap_token_scores = df[df["comment_id"] == comment_id][["token", "shap_weight"]].to_dict(orient="records")
    ig_token_scores = df[df["comment_id"] == comment_id][["token", "ig_weight"]].to_dict(orient="records")
    deeplift_token_scores = df[df["comment_id"] == comment_id][["token", "deeplift_weight"]].to_dict(orient="records")
    exai4_mean_scores = df[df["comment_id"] == comment_id][["token", "exai4_mean"]].to_dict(orient="records")
    exai4_median_scores = df[df["comment_id"] == comment_id][["token", "exai4_median"]].to_dict(orient="records")
    exai3_mean_scores = df[df["comment_id"] == comment_id][["token", "exai3_mean"]].to_dict(orient="records")
    exai3_median_scores = df[df["comment_id"] == comment_id][["token", "exai3_median"]].to_dict(orient="records")
    exai2_mean_scores = df[df["comment_id"] == comment_id][["token", "exai2_mean"]].to_dict(orient="records")

    # Convert to list of tuples instead of dicts, skipping NaNs
    lime_token_scores = [(item["token"], item["lime_weight"]) for item in lime_token_scores if not pd.isna(item["lime_weight"])]
    shap_token_scores = [(item["token"], item["shap_weight"]) for item in shap_token_scores if not pd.isna(item["shap_weight"])]
    ig_token_scores = [(item["token"], item["ig_weight"]) for item in ig_token_scores if not pd.isna(item["ig_weight"])]
    deeplift_token_scores = [(item["token"], item["deeplift_weight"]) for item in deeplift_token_scores if not pd.isna(item["deeplift_weight"])]
    exai4_mean_scores = [(item["token"], item["exai4_mean"]) for item in exai4_mean_scores if not pd.isna(item["exai4_mean"])]
    exai4_median_scores = [(item["token"], item["exai4_median"]) for item in exai4_median_scores if not pd.isna(item["exai4_median"])]
    exai3_mean_scores = [(item["token"], item["exai3_mean"]) for item in exai3_mean_scores if not pd.isna(item["exai3_mean"])]
    exai3_median_scores = [(item["token"], item["exai3_median"]) for item in exai3_median_scores if not pd.isna(item["exai3_median"])]
    exai2_mean_scores = [(item["token"], item["exai2_mean"]) for item in exai2_mean_scores if not pd.isna(item["exai2_mean"])]

    # Pass lists of tuples to top_tokens_elbow
    lime_top_tokens = top_tokens_elbow(lime_token_scores)
    shap_top_tokens = top_tokens_elbow(shap_token_scores)
    ig_top_tokens = top_tokens_elbow(ig_token_scores)
    deeplift_top_tokens = top_tokens_elbow(deeplift_token_scores)
    exai4_mean_top_tokens = top_tokens_elbow(exai4_mean_scores)
    exai4_median_top_tokens = top_tokens_elbow(exai4_median_scores)
    exai3_mean_top_tokens = top_tokens_elbow(exai3_mean_scores)
    exai3_median_top_tokens = top_tokens_elbow(exai3_median_scores)
    exai2_mean_top_tokens = top_tokens_elbow(exai2_mean_scores)

    # Mark top tokens in the DataFrame
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(lime_top_tokens)), "lime_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(shap_top_tokens)), "shap_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(ig_top_tokens)), "ig_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(deeplift_top_tokens)), "deeplift_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(exai4_mean_top_tokens)), "exai4_mean_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(exai4_median_top_tokens)), "exai4_median_hard"] = 1 
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(exai3_mean_top_tokens)), "exai3_mean_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(exai3_median_top_tokens)), "exai3_median_hard"] = 1
    df.loc[(df["comment_id"] == comment_id) & (df["token"].isin(exai2_mean_top_tokens)), "exai2_mean_hard"] = 1

Running elbow on explanations:   0%|          | 0/303 [00:00<?, ?comment/s]

In [34]:
df.to_csv("data/plausibility/xai_res_with_annotations_" + model_pred_col + ".csv", index=False)

In [35]:
model_pred_col = "Camelbert-MSA"
model_pred_col2 = "AraBert"

In [36]:
df = pd.read_csv("data/plausibility/xai_res_with_annotations_" + model_pred_col + ".csv")
df2 = pd.read_csv("data/plausibility/xai_res_with_annotations_" + model_pred_col2 + ".csv")

In [37]:
methods_hard_cols = ["lime_hard", "shap_hard", "ig_hard", "deeplift_hard", "exai4_mean_hard", "exai4_median_hard", "exai3_mean_hard", "exai3_median_hard", "exai2_mean_hard"]

In [38]:
df["human_annot"] = df["human_annot"].astype(int)
plausibility_metrics = {}
for method_col in methods_hard_cols:
    accuracy = accuracy_score(df["human_annot"], df[method_col])
    precision = precision_score(df["human_annot"], df[method_col])
    recall = recall_score(df["human_annot"], df[method_col])
    f1 = f1_score(df["human_annot"], df[method_col])
    jaccard = jaccard_score(df["human_annot"], df[method_col])
    plausibility_metrics[method_col] = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "jaccard": jaccard
    }

In [39]:
plausibility_metrics_df = pd.DataFrame(plausibility_metrics).T
plausibility_metrics_df

,accuracy,precision,recall,f1,jaccard
lime_hard,0.395096,0.259691,0.735470,0.383847,0.237507
shap_hard,0.493648,0.285120,0.647832,0.395968,0.246858
ig_hard,0.355628,0.265357,0.856780,0.405214,0.254087
deeplift_hard,0.316337,0.261331,0.913515,0.406402,0.255022
exai4_mean_hard,0.362659,0.267431,0.855397,0.407470,0.255864
exai4_median_hard,0.364195,0.266923,0.848478,0.406093,0.254778
exai3_mean_hard,0.413530,0.275068,0.788284,0.407827,0.256145
exai3_median_hard,0.386647,0.269328,0.813884,0.404725,0.253702
exai2_mean_hard,0.430428,0.276617,0.757380,0.405232,0.254101


In [40]:
df2["human_annot"] = df2["human_annot"].astype(int)
plausibility_metrics2 = {}
for method_col in methods_hard_cols:
    accuracy = accuracy_score(df2["human_annot"], df2[method_col])
    precision = precision_score(df2["human_annot"], df2[method_col])
    recall = recall_score(df2["human_annot"], df2[method_col])
    f1 = f1_score(df2["human_annot"], df2[method_col])
    jaccard = jaccard_score(df2["human_annot"], df2[method_col])
    plausibility_metrics2[method_col] = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "jaccard": jaccard
    }

In [41]:
plausibility_metrics_df2 = pd.DataFrame(plausibility_metrics2).T
plausibility_metrics_df2

,accuracy,precision,recall,f1,jaccard
lime_hard,0.413139,0.265629,0.743013,0.391350,0.243278
shap_hard,0.544216,0.313832,0.670047,0.427456,0.271824
ig_hard,0.333752,0.255821,0.850606,0.393343,0.244821
deeplift_hard,0.305678,0.253688,0.893149,0.395141,0.246216
exai4_mean_hard,0.434305,0.280665,0.785555,0.413569,0.260691
exai4_median_hard,0.415903,0.274976,0.794460,0.408547,0.256714
exai3_mean_hard,0.487062,0.288946,0.698244,0.408745,0.256870
exai3_median_hard,0.434744,0.272802,0.736087,0.398074,0.248497
exai2_mean_hard,0.500942,0.297751,0.710611,0.419661,0.265551


In [42]:
plausibility_metrics_df_avg = (plausibility_metrics_df + plausibility_metrics_df2) / 2
plausibility_metrics_df_avg

,accuracy,precision,recall,f1,jaccard
lime_hard,0.404118,0.262660,0.739242,0.387598,0.240392
shap_hard,0.518932,0.299476,0.658940,0.411712,0.259341
ig_hard,0.344690,0.260589,0.853693,0.399279,0.249454
deeplift_hard,0.311007,0.257510,0.903332,0.400772,0.250619
exai4_mean_hard,0.398482,0.274048,0.820476,0.410520,0.258277
exai4_median_hard,0.390049,0.270950,0.821469,0.407320,0.255746
exai3_mean_hard,0.450296,0.282007,0.743264,0.408286,0.256507
exai3_median_hard,0.410696,0.271065,0.774985,0.401399,0.251100
exai2_mean_hard,0.465685,0.287184,0.733996,0.412447,0.259826
